In [1]:
import pandas as pd

master = pd.read_csv("/Users/aliabonanno/public_policy_project/master_panel_monthly_fc.csv",
                     dtype={"FIPS": str})

counties = master.drop_duplicates(subset=["FIPS"]).copy()
counties["state"] = counties["FIPS"].str[:2]

# Cooperative share by state
state_coop = (counties.groupby("state")
              .agg(total_counties=("FIPS", "count"),
                   coop_counties=("is_coop", "sum"),
                   mean_poverty=("pct_below_150pct_poverty", "mean"))
              .assign(pct_coop=lambda x: x["coop_counties"]/x["total_counties"])
              .reset_index()
              .sort_values("pct_coop", ascending=False))

print(state_coop.head(20).to_string())

/Users/aliabonanno/Downloads/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


   state  total_counties  coop_counties  mean_poverty  pct_coop
1     02              14             11      0.254773  0.785714
42    47              77             47      0.311379  0.610390
16    20              83             28      0.231330  0.337349
50    56              21              5      0.190085  0.238095
41    46              60             12      0.260236  0.200000
11    15               5              1      0.198345  0.200000
26    30              53              6      0.269046  0.113208
17    21             111             12      0.335823  0.108108
45    50              14              1      0.209781  0.071429
43    48             113              8      0.294405  0.070796
27    31              73              5      0.220835  0.068493
5     08              60              4      0.249762  0.066667
28    32              16              1      0.237222  0.062500
24    28              69              4      0.397702  0.057971
20    24              21              1 

In [3]:
import pandas as pd

master = pd.read_csv("/Users/aliabonanno/public_policy_project/master_panel_monthly_fc.csv",
                     dtype={"FIPS": str})

# State FIPS to name crosswalk
state_names = {
    "01": "Alabama", "02": "Alaska", "04": "Arizona", "05": "Arkansas",
    "06": "California", "08": "Colorado", "09": "Connecticut", "10": "Delaware",
    "11": "DC", "12": "Florida", "13": "Georgia", "15": "Hawaii",
    "16": "Idaho", "17": "Illinois", "18": "Indiana", "19": "Iowa",
    "20": "Kansas", "21": "Kentucky", "22": "Louisiana", "23": "Maine",
    "24": "Maryland", "25": "Massachusetts", "26": "Michigan", "27": "Minnesota",
    "28": "Mississippi", "29": "Missouri", "30": "Montana", "31": "Nebraska",
    "32": "Nevada", "33": "New Hampshire", "34": "New Jersey", "35": "New Mexico",
    "36": "New York", "37": "North Carolina", "38": "North Dakota", "39": "Ohio",
    "40": "Oklahoma", "41": "Oregon", "42": "Pennsylvania", "44": "Rhode Island",
    "45": "South Carolina", "46": "South Dakota", "47": "Tennessee", "48": "Texas",
    "49": "Utah", "50": "Vermont", "51": "Virginia", "53": "Washington",
    "54": "West Virginia", "55": "Wisconsin", "56": "Wyoming"
}

counties = master.drop_duplicates(subset=["FIPS"]).copy()
counties["state_fips"] = counties["FIPS"].str[:2]
counties["state_name"] = counties["state_fips"].map(state_names)

state_coop = (counties.groupby(["state_fips", "state_name"])
              .agg(total_counties=("FIPS", "count"),
                   coop_counties=("is_coop", "sum"),
                   mean_poverty=("pct_below_150pct_poverty", "mean"))
              .assign(pct_coop=lambda x: x["coop_counties"]/x["total_counties"])
              .reset_index()
              .sort_values("pct_coop", ascending=False))

print(state_coop[state_coop["pct_coop"] > 0][
    ["state_name", "total_counties", "coop_counties", 
     "mean_poverty", "pct_coop"]
].to_string())

# Save for LIHEAP merge
state_coop.to_csv(
    "/Users/aliabonanno/public_policy_project/state_coop_concentration.csv",
    index=False)
print("\nSaved state_coop_concentration.csv")

      state_name  total_counties  coop_counties  mean_poverty  pct_coop
1         Alaska              14             11      0.254773  0.785714
42     Tennessee              77             47      0.311379  0.610390
16        Kansas              83             28      0.231330  0.337349
50       Wyoming              21              5      0.190085  0.238095
41  South Dakota              60             12      0.260236  0.200000
11        Hawaii               5              1      0.198345  0.200000
26       Montana              53              6      0.269046  0.113208
17      Kentucky             111             12      0.335823  0.108108
45       Vermont              14              1      0.209781  0.071429
43         Texas             113              8      0.294405  0.070796
27      Nebraska              73              5      0.220835  0.068493
5       Colorado              60              4      0.249762  0.066667
28        Nevada              16              1      0.237222  0

In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv("/Users/aliabonanno/public_policy_project/liheap.csv")
df.columns = ["year", "state_name", "pct_served_str"]
df["pct_served"] = df["pct_served_str"].str.replace("%","").astype(float)
liheap = df[df["year"] == "2022"].copy()

state_coop = {
    "Alaska": 0.786, "Tennessee": 0.610, "Kansas": 0.337,
    "Wyoming": 0.238, "South Dakota": 0.200, "Hawaii": 0.200,
    "Montana": 0.113, "Kentucky": 0.108, "Vermont": 0.071,
    "Texas": 0.071, "Nebraska": 0.068, "Colorado": 0.067,
    "Nevada": 0.063, "Mississippi": 0.058, "Maryland": 0.048,
    "Virginia": 0.047, "Utah": 0.037, "New Mexico": 0.037,
    "Michigan": 0.036, "Minnesota": 0.035, "Idaho": 0.023,
    "North Dakota": 0.019, "Alabama": 0.016, "Oklahoma": 0.013,
    "Arkansas": 0.013,
}

liheap["pct_coop"] = liheap["state_name"].map(state_coop).fillna(0)
liheap["no_cooling"] = (liheap["pct_served"] == 0).astype(int)
liheap["high_coop"] = (liheap["pct_coop"] >= 0.10).astype(int)

print("Share of states offering zero cooling assistance:")
print(liheap.groupby("high_coop")["no_cooling"].agg(
    n_states="count",
    n_zero="sum",
    pct_zero="mean"
).round(3))

print("\nMean coop concentration:")
print(f"  States with zero cooling: {liheap[liheap['no_cooling']==1]['pct_coop'].mean():.3f}")
print(f"  States with cooling:      {liheap[liheap['no_cooling']==0]['pct_coop'].mean():.3f}")

Share of states offering zero cooling assistance:
           n_states  n_zero  pct_zero
high_coop                            
0                43      20     0.465
1                 8       6     0.750

Mean coop concentration:
  States with zero cooling: 0.085
  States with cooling:      0.044


In [7]:
import pandas as pd

urdb = pd.read_csv("/Users/aliabonanno/public_policy_project/OpenEI/usurdb.csv.gz",
                   compression="gzip", low_memory=False)

print("All columns:")
print(urdb.columns.tolist())

# Look for TOU/peak pricing indicators
tou_cols = [c for c in urdb.columns if any(x in c.lower() 
            for x in ["tou", "time", "peak", "demand", "period", "tier"])]
print(f"\nPotential TOU/peak columns: {tou_cols}")

# Filter to residential
urdb_res = urdb[urdb["sector"].str.lower().str.contains("residential", na=False)]
print(f"\nResidential rates: {len(urdb_res)}")
print(f"\nSample of rate names:")
print(urdb_res["name"].dropna().head(20).tolist())

All columns:
['label', 'eiaid', 'name', 'is_default', 'startdate', 'enddate', 'latest_update', 'utility', 'sector', 'description', 'servicetype', 'source', 'sourceparent', 'dgrules', 'peakkwcapacitymin', 'peakkwcapacitymax', 'peakkwcapacityhistory', 'peakkwhusagemin', 'peakkwhusagemax', 'peakkwhusagehistory', 'voltagecategory', 'phasewiring', 'voltageminimum', 'voltagemaximum', 'fixedchargefirstmeter', 'fixedchargeeaaddl', 'fixedchargeunits', 'mincharge', 'minchargeunits', 'fixedattrs', 'basicinformationcomments', 'flatdemandunit', 'flatdemandstructure/period0/tier0max', 'flatdemandstructure/period0/tier0rate', 'flatdemandstructure/period0/tier0adj', 'flatdemandstructure/period0/tier0unit', 'flatdemandstructure/period0/tier1max', 'flatdemandstructure/period0/tier1rate', 'flatdemandstructure/period0/tier1adj', 'flatdemandstructure/period0/tier1unit', 'flatdemandstructure/period0/tier2max', 'flatdemandstructure/period0/tier2rate', 'flatdemandstructure/period0/tier2adj', 'flatdemandstruct

In [9]:
import pandas as pd
import numpy as np

urdb = pd.read_csv("/Users/aliabonanno/public_policy_project/OpenEI/usurdb.csv.gz",
                   compression="gzip", low_memory=False)

urdb_res = urdb[urdb["sector"].str.lower().str.contains("residential", na=False)].copy()
urdb_res["eiaid"] = pd.to_numeric(urdb_res["eiaid"], errors="coerce")
urdb_res = urdb_res.dropna(subset=["eiaid"])
urdb_res["eiaid"] = urdb_res["eiaid"].astype(int)

# TOU indicator: has a second energy rate period
urdb_res["has_tou"] = urdb_res["energyratestructure/period1/tier0rate"].notna().astype(int)

# Also check rate name for TOU keywords
tou_keywords = ["time-of-use", "time of use", "tou", "time-of-day", 
                "time of day", "peak", "off-peak", "on-peak"]
name_lower = urdb_res["name"].fillna("").str.lower()
urdb_res["name_tou"] = name_lower.apply(
    lambda x: any(k in x for k in tou_keywords)).astype(int)

# Combined TOU flag
urdb_res["is_tou"] = ((urdb_res["has_tou"] == 1) | 
                       (urdb_res["name_tou"] == 1)).astype(int)

# Aggregate to utility level — does utility have ANY TOU residential rate?
utility_tou = (urdb_res.groupby("eiaid")
               .agg(n_rates=("is_tou","count"),
                    n_tou=("is_tou","sum"))
               .assign(has_any_tou=lambda x: (x["n_tou"] > 0).astype(int))
               .reset_index())

print(f"Utilities with any TOU residential rate: "
      f"{utility_tou['has_any_tou'].sum()} of {len(utility_tou)} "
      f"({utility_tou['has_any_tou'].mean():.1%})")

# Merge to EIA sales to get ownership type
sales = pd.read_csv("/Users/aliabonanno/public_policy_project/eia861_sales.csv")
sales["utility_id"] = pd.to_numeric(sales["utility_id"], errors="coerce")
sales_tou = sales.merge(utility_tou[["eiaid","has_any_tou"]], 
                        left_on="utility_id", right_on="eiaid", how="left")
sales_tou["has_any_tou"] = sales_tou["has_any_tou"].fillna(0)

utilities = sales_tou.drop_duplicates(subset=["utility_id"])

print("\nTOU prevalence by ownership type:")
print(utilities.groupby("ownership")["has_any_tou"]
      .agg(n_utilities="count", n_tou="sum", pct_tou="mean")
      .round(3).sort_values("pct_tou", ascending=False))

Utilities with any TOU residential rate: 1027 of 2652 (38.7%)

TOU prevalence by ownership type:
                             n_utilities  n_tou  pct_tou
ownership                                               
Investor Owned                       164  115.0    0.701
Cooperative                          815  315.0    0.387
Municipal                           1773  553.0    0.312
State                                 10    3.0    0.300
Political Subdivision                 92   24.0    0.261
Community Choice Aggregator           26    1.0    0.038
Retail Power Marketer                351    2.0    0.006
Behind the Meter                      15    0.0    0.000
Federal                                6    0.0    0.000


In [11]:
import pandas as pd
import numpy as np

# You already have utility_tou from above
# Merge to territory to get county-level TOU prevalence

territory = pd.read_csv("/Users/aliabonanno/public_policy_project/eia861_territory.csv")
sales = pd.read_csv("/Users/aliabonanno/public_policy_project/eia861_sales.csv")
sales["utility_id"] = pd.to_numeric(sales["utility_id"], errors="coerce")

sales_tou = sales.merge(utility_tou[["eiaid","has_any_tou"]],
                        left_on="utility_id", right_on="eiaid", how="left")
sales_tou["has_any_tou"] = sales_tou["has_any_tou"].fillna(0)

# Keep only traditional utility types
keep = ["Investor Owned","Cooperative","Municipal",
        "Political Subdivision","State","Federal"]
sales_tou = sales_tou[sales_tou["ownership"].isin(keep)]

# Customer-weighted TOU prevalence per county
# Need territory crosswalk for county FIPS
# Use same territory_fips from master merge
import os
OUTPUT_DIR = "/Users/aliabonanno/public_policy_project"

fips_ref = pd.read_csv(os.path.join(OUTPUT_DIR,"county_fips.csv"),
                       sep="|", dtype=str)
fips_ref.columns = [c.strip().lower() for c in fips_ref.columns]
fips_ref["FIPS"] = fips_ref["statefp"].str.zfill(2) + fips_ref["countyfp"].str.zfill(3)

def clean_county(name):
    if pd.isna(name): return name
    import re
    s = re.sub(r"\s+(County|Parish|Borough|Municipality|Census Area|"
               r"City and Borough|Municipio|District|Island|city|City)$",
               "", str(name))
    s = s.strip().lower()
    s = s.replace("st. ","saint ").replace("st ","saint ").replace(".","")
    return s.strip()

territory["county_clean"] = territory["county_name"].apply(clean_county)
territory["state_abbr"] = territory["state"].str.upper()
fips_ref["county_clean"] = fips_ref["countyname"].apply(clean_county)
fips_ref["state_abbr"] = fips_ref["state"].str.upper()

territory_fips = territory.merge(
    fips_ref[["state_abbr","county_clean","FIPS"]],
    on=["state_abbr","county_clean"], how="left")

utility_county = sales_tou.merge(
    territory_fips[["utility_id","year","state","FIPS"]],
    on=["utility_id","year"], how="inner").dropna(subset=["FIPS"])
utility_county["FIPS"] = utility_county["FIPS"].str.zfill(5)

# TOU x customers
utility_county["tou_x_customers"] = (utility_county["has_any_tou"] * 
                                      utility_county["res_customers"])

county_tou = (utility_county.groupby("FIPS")
              .agg(tou_x_cust_sum=("tou_x_customers","sum"),
                   cust_sum=("res_customers","sum"))
              .reset_index())
county_tou["tou_prevalence"] = (county_tou["tou_x_cust_sum"] / 
                                 county_tou["cust_sum"])

print(f"County TOU prevalence: mean={county_tou['tou_prevalence'].mean():.3f}")
print(county_tou["tou_prevalence"].describe().round(3))

# Check by ownership type
master = pd.read_csv(f"{OUTPUT_DIR}/master_panel_monthly_fc.csv",
                     dtype={"FIPS":str})
counties = master.drop_duplicates(subset=["FIPS"])
merged = counties.merge(county_tou, on="FIPS", how="left")

print("\nMean TOU prevalence by ownership type:")
print(merged.groupby("dominant_ownership_type")["tou_prevalence"]
      .agg(["mean","median","count"]).round(3))

county_tou.to_csv(f"{OUTPUT_DIR}/county_tou_prevalence.csv", index=False)
print("\nSaved county_tou_prevalence.csv")

County TOU prevalence: mean=0.789
count    3115.000
mean        0.789
std         0.355
min         0.000
25%         0.813
50%         0.980
75%         1.000
max         1.000
Name: tou_prevalence, dtype: float64

Mean TOU prevalence by ownership type:
                          mean  median  count
dominant_ownership_type                      
Cooperative              0.484   0.534    165
IOU                      0.860   0.986   2532
Municipal                0.746   0.911    117

Saved county_tou_prevalence.csv


In [15]:
import pandas as pd
import numpy as np

master = pd.read_csv("/Users/aliabonanno/public_policy_project/master_panel_monthly_fc.csv",
                     dtype={"FIPS": str})

# Energy burden proxy: monthly bill / monthly income
# Monthly bill = exp(log_implied_price) * exp(log_res_sales_per_customer) * 1000
# Monthly income = median_hh_income / 12
master["monthly_bill"] = (np.exp(master["log_implied_price"]) *
                           np.exp(master["log_res_sales_per_customer"]))
master["monthly_income"] = master["median_hh_income"] / 12
master["energy_burden"] = master["monthly_bill"] / master["monthly_income"]

# By ownership type
print("Mean energy burden by ownership type:")
print(master.groupby("dominant_ownership_type")["energy_burden"]
      .agg(["mean","median","count"]).round(4))

# By poverty quartile within each ownership type
master["pov_quartile"] = pd.qcut(master["pct_below_150pct_poverty"], 4,
                                  labels=["Q1","Q2","Q3","Q4"])
print("\nEnergy burden by ownership x poverty quartile:")
print(master.groupby(["dominant_ownership_type","pov_quartile"])["energy_burden"]
      .mean().unstack().round(4))

Mean energy burden by ownership type:
                           mean  median   count
dominant_ownership_type                        
Cooperative              0.0323  0.0303   19308
IOU                      0.0272  0.0245  302267
Municipal                0.0250  0.0236   13956

Energy burden by ownership x poverty quartile:
pov_quartile                 Q1      Q2      Q3      Q4
dominant_ownership_type                                
Cooperative              0.0217  0.0267  0.0327  0.0436
IOU                      0.0188  0.0232  0.0283  0.0382
Municipal                0.0206  0.0249  0.0293  0.0342


In [18]:
p10 = master["pct_below_150pct_poverty"].quantile(0.10)
p90 = master["pct_below_150pct_poverty"].quantile(0.90)
pov_deviation = p90 - p10  # was 0.214 per your slide

In [20]:
pov_deviation

0.2136979255442914

In [22]:
import numpy as np

# Coefficients from Spec 5
coef_b8 = -0.025
coef_b9 = -0.038

# From your earlier output
mean_b8 = 11.91
mean_b9 = 1.05
mean_sales_per_customer_mwh = 0.896  # MWh per customer per month
implied_price = 126.76  # $/MWh
pov_deviation = 0.2137  # P90 - P10

# Coop-only shortfall per customer per summer month
log_shortfall = (coef_b8 * mean_b8 + coef_b9 * mean_b9) * pov_deviation
pct_shortfall = (np.exp(log_shortfall) - 1) * 100
mwh_shortfall = abs(log_shortfall) * mean_sales_per_customer_mwh
dollar_shortfall = mwh_shortfall * implied_price

print(f"Log-point reduction: {log_shortfall:.4f}")
print(f"% reduction: {pct_shortfall:.2f}%")
print(f"MWh shortfall per customer per month: {mwh_shortfall:.4f}")
print(f"Dollar shortfall per customer per month: ${dollar_shortfall:.2f}")

# 3 summer months
total_customers = 29508342
total_dollar_coop = dollar_shortfall * total_customers * 3
kwh_per_summer = mwh_shortfall * 3 * 1000
ac_hours = kwh_per_summer / 3.5

print(f"\nkWh foregone per household per summer (coop only): {kwh_per_summer:.1f}")
print(f"AC hours foregone per summer (coop only): {ac_hours:.1f}")
print(f"Aggregate coop-only shortfall: ${total_dollar_coop/1e6:.1f}M")

Log-point reduction: -0.0722
% reduction: -6.96%
MWh shortfall per customer per month: 0.0647
Dollar shortfall per customer per month: $8.20

kWh foregone per household per summer (coop only): 194.0
AC hours foregone per summer (coop only): 55.4
Aggregate coop-only shortfall: $725.5M
